# ML Dynamic Pricing - Phase 2: Elasticity-Based Optimization

Combines rule-based pricing with price elasticity modeling for revenue-optimal recommendations.

## Phase 1 Features (Rule-Based)
- **Age-based markdowns**: Products markdown based on days since receipt
- **Inventory triggers**: Overstocked items get price reductions
- **Seasonal clearance**: End-of-season products marked down

## Phase 2 Features (NEW)
- **Elasticity integration**: Uses price elasticity coefficients from a configurable Gold dependency table
- **Revenue optimization**: Calculates optimal price point to maximize revenue
- **Inventory urgency**: Adjusts recommendations based on stock levels and sell-through rate
- **Revenue impact**: Estimates expected revenue change per recommendation
- **Confidence levels**: Provides confidence scores based on data quality and elasticity estimates

## Constraints
- Min Margin: configurable via `MIN_MARGIN_PCT`
- Max Price: configurable via `MAX_PRICE_FACTOR`
- Max Change/Week: configurable via `MAX_CHANGE_PCT_WEEKLY`
- Min Duration: configurable via `MIN_DURATION_DAYS`

## Data Flow
```
Configured Silver source tables
  + configured Gold elasticity dependency
  --> configured Gold pricing outputs
```

## Usage
Run this notebook on-demand or schedule it at the cadence that fits your pricing process.

## Output Schema
The configured pricing recommendations table includes:
- Current price and recommended price
- Reason codes for transparency
- Constraint validation flags
- Price elasticity coefficient and category
- Expected revenue impact
- Confidence level (high/medium/low)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone, timedelta
import os
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
def get_env_list(var_name, default=None):
    value = get_env(var_name, default)
    if value is None:
        return []
    return [item.strip().lower() for item in value.split(",") if item.strip()]

SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="dynamic_pricing")
PRODUCTS_TABLE = get_env("PRODUCTS_TABLE", default="dim_products")
INVENTORY_TXN_TABLE = get_env("INVENTORY_TXN_TABLE", default="fact_store_inventory_txn")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
PRICE_ELASTICITY_TABLE = get_env("PRICE_ELASTICITY_TABLE", default="price_elasticity")
PRICING_CONSTRAINTS_TABLE = get_env("PRICING_CONSTRAINTS_TABLE", default="pricing_constraints")
PRICING_RECOMMENDATIONS_TABLE = get_env("PRICING_RECOMMENDATIONS_TABLE", default="pricing_recommendations")
PRICE_ELASTICITY_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRICE_ELASTICITY_TABLE}"
PRICING_CONSTRAINTS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRICING_CONSTRAINTS_TABLE}"
PRICING_RECOMMENDATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRICING_RECOMMENDATIONS_TABLE}"

# Pricing constraints (business rules)
MIN_MARGIN_PCT = 0.15  # 15% minimum margin
MAX_PRICE_FACTOR = 1.0  # Max price = MSRP
MAX_CHANGE_PCT_WEEKLY = 0.10  # Max 10% change per week
MIN_DURATION_DAYS = 7  # Min 7 days between price changes

# Markdown rules (Phase 1)
AGE_THRESHOLD_MODERATE = 30  # Days - start moderate markdown
AGE_THRESHOLD_AGGRESSIVE = 60  # Days - aggressive markdown
AGE_MARKDOWN_MODERATE = 0.10  # 10% markdown
AGE_MARKDOWN_AGGRESSIVE = 0.25  # 25% markdown

INVENTORY_HIGH_THRESHOLD = 100  # Units - considered overstocked
INVENTORY_MARKDOWN = 0.15  # 15% markdown for overstocked

SALES_WINDOW_DAYS = int(get_env("SALES_WINDOW_DAYS", default="30"))
# Seasonal clearance configuration
CLEARANCE_MONTHS = [int(month.strip()) for month in get_env("CLEARANCE_MONTHS", default="1,2,3").split(",") if month.strip()]


CLEARANCE_TAGS = get_env_list("CLEARANCE_TAGS", default="winter,holiday")
CLEARANCE_MARKDOWN = 0.30  # 30% markdown for clearance

# Phase 2: Elasticity-based optimization parameters
ELASTICITY_WEIGHT = 0.6  # Weight for elasticity-based price vs rule-based
INVENTORY_URGENCY_DAYS = 14  # Days of inventory to trigger urgency
MIN_OBSERVATIONS_FOR_ELASTICITY = 20  # Min data points to use elasticity

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {PRODUCTS_TABLE}, {INVENTORY_TXN_TABLE}, {RECEIPT_LINES_TABLE}")
print(f"Gold dependency/output tables: {PRICE_ELASTICITY_TABLE_NAME}, {PRICING_CONSTRAINTS_TABLE_NAME}, {PRICING_RECOMMENDATIONS_TABLE_NAME}")
print(f"Constraints: Min Margin={MIN_MARGIN_PCT*100}%, Max Price factor={MAX_PRICE_FACTOR}, Max Change/Week={MAX_CHANGE_PCT_WEEKLY*100}%")
print(f"Sales window days={SALES_WINDOW_DAYS}, Clearance months={CLEARANCE_MONTHS}, Clearance tags={CLEARANCE_TAGS}")
print(f"Phase 2: ELASTICITY_WEIGHT={ELASTICITY_WEIGHT}, INVENTORY_URGENCY_DAYS={INVENTORY_URGENCY_DAYS}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def read_gold(table_name):
    try:
        return spark.table(f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}")
    except AnalysisException:
        return None

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"  {full_name}: {df.count()} rows")

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def resolve_table_column(df, table_name, *candidates):
    columns_by_lower = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}. Available columns: {df.columns}"
    )

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)


In [ ]:
# =============================================================================
# PRICING CONSTRAINTS TABLE
# =============================================================================

print("="*60)
print("CREATING PRICING CONSTRAINTS TABLE")
print("="*60)

# Create constraints reference table for transparency and configurability
constraints_data = [
    {"constraint_name": "min_margin_pct", "constraint_value": MIN_MARGIN_PCT, "description": f"Minimum margin percentage (price >= cost * (1 + {MIN_MARGIN_PCT}))"},
    {"constraint_name": "max_price_factor", "constraint_value": MAX_PRICE_FACTOR, "description": f"Maximum price as factor of MSRP ({MAX_PRICE_FACTOR} = current setting)"},
    {"constraint_name": "max_change_pct_weekly", "constraint_value": MAX_CHANGE_PCT_WEEKLY, "description": "Maximum price change per week (as percentage)"},
    {"constraint_name": "min_duration_days", "constraint_value": float(MIN_DURATION_DAYS), "description": "Minimum days between price changes"},
    {"constraint_name": "age_threshold_moderate", "constraint_value": float(AGE_THRESHOLD_MODERATE), "description": "Days to trigger moderate markdown"},
    {"constraint_name": "age_threshold_aggressive", "constraint_value": float(AGE_THRESHOLD_AGGRESSIVE), "description": "Days to trigger aggressive markdown"},
    {"constraint_name": "age_markdown_moderate", "constraint_value": AGE_MARKDOWN_MODERATE, "description": "Markdown percentage for moderate age threshold"},
    {"constraint_name": "age_markdown_aggressive", "constraint_value": AGE_MARKDOWN_AGGRESSIVE, "description": "Markdown percentage for aggressive age threshold"},
    {"constraint_name": "inventory_high_threshold", "constraint_value": float(INVENTORY_HIGH_THRESHOLD), "description": "Units threshold for overstocked items"},
    {"constraint_name": "inventory_markdown", "constraint_value": INVENTORY_MARKDOWN, "description": "Markdown percentage for overstocked items"},
    {"constraint_name": "clearance_markdown", "constraint_value": CLEARANCE_MARKDOWN, "description": "Markdown percentage for seasonal clearance"},
    {"constraint_name": "elasticity_weight", "constraint_value": ELASTICITY_WEIGHT, "description": "Weight for elasticity-based pricing (Phase 2)"},
    {"constraint_name": "inventory_urgency_days", "constraint_value": float(INVENTORY_URGENCY_DAYS), "description": "Days of inventory to trigger urgency (Phase 2)"},
]

df_constraints = spark.createDataFrame(constraints_data)
save_gold(df_constraints, PRICING_CONSTRAINTS_TABLE)
print()

In [ ]:
# =============================================================================
# LOAD SOURCE DATA
# =============================================================================

print("="*60)
print("LOADING SOURCE DATA")
print("="*60)

if not silver_exists(PRODUCTS_TABLE):
    raise Exception(f"{PRODUCTS_TABLE} table not found. Run 02-historical-data-load.ipynb first.")

# Load product master with pricing data
df_products_raw = read_silver(PRODUCTS_TABLE)
product_id_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_id", "id", "ID")
product_name_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_name", "ProductName")
department_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "department", "Department")
category_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "category", "Category")
subcategory_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "subcategory", "Subcategory")
cost_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "cost", "Cost")
msrp_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "msrp", "MSRP")
sale_price_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "sale_price", "SalePrice")
tags_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "tags", "Tags")
df_products = df_products_raw.select(
    F.col(product_id_col).alias("product_id"),
    F.col(product_name_col).alias("ProductName"),
    F.col(department_col).alias("Department"),
    F.col(category_col).alias("Category"),
    F.col(subcategory_col).alias("Subcategory"),
    F.col(cost_col).cast("double").alias("Cost"),
    F.col(msrp_col).cast("double").alias("MSRP"),
    F.col(sale_price_col).cast("double").alias("SalePrice"),
    F.col(tags_col).alias("Tags")
)

print(f"Products loaded: {df_products.count()}")

# Load current inventory positions (aggregated across all stores)
df_inventory = None
if silver_exists(INVENTORY_TXN_TABLE):
    # Calculate latest inventory snapshot per store/product, then roll up to product level
    window_spec = Window.partitionBy("store_id", "product_id").orderBy(F.desc("event_ts"))
    df_inventory = (
        read_silver(INVENTORY_TXN_TABLE)
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .groupBy("product_id")
        .agg(
            F.sum("balance").alias("total_inventory"),
            F.max("event_ts").alias("inventory_as_of")
        )
    )
    print(f"Inventory positions loaded: {df_inventory.count()}")
else:
    print("No inventory data available - using product data only")

# Establish a pricing reference date anchored to the latest available sales activity when present
pricing_reference_date = datetime.now(timezone.utc).date()
df_receipt_lines_source = None
if silver_exists(RECEIPT_LINES_TABLE):
    df_receipt_lines_source = (
        read_silver(RECEIPT_LINES_TABLE)
        .select(
            "product_id",
            "quantity",
            F.col("event_ts").cast("timestamp").alias("event_ts")
        )
        .withColumn("event_date", F.to_date("event_ts"))
        .cache()
    )
    receipt_line_source_count = df_receipt_lines_source.count()
    latest_sales_row = df_receipt_lines_source.agg(F.max("event_date").alias("latest_sales_date")).first()
    if latest_sales_row["latest_sales_date"] is not None:
        pricing_reference_date = latest_sales_row["latest_sales_date"]
    print(f"Receipt lines source loaded: {receipt_line_source_count}")
else:
    print("Receipt lines not available - using current UTC date for pricing reference")

pricing_reference_date_lit = F.lit(pricing_reference_date).cast("date")
print(f"Pricing reference date: {pricing_reference_date}")

# Load previous pricing recommendations to check duration constraint
df_previous_prices = read_gold(PRICING_RECOMMENDATIONS_TABLE)
if df_previous_prices is not None:
    required_previous_price_columns = {"product_id", "recommended_price"}
    if not required_previous_price_columns.issubset(set(df_previous_prices.columns)):
        print("Previous pricing recommendations missing required columns - skipping previous price constraints")
        df_previous_prices = None
    elif "recommendation_date" not in df_previous_prices.columns and "recommendation_ts" not in df_previous_prices.columns:
        print("Previous pricing recommendations missing recommendation date/timestamp - skipping previous price constraints")
        df_previous_prices = None
    else:
        print(f"Previous pricing recommendations loaded: {df_previous_prices.count()}")
else:
    print("No previous pricing recommendations - first run")

# Phase 2: Load price elasticity data
df_elasticity = read_gold(PRICE_ELASTICITY_TABLE)
if df_elasticity is not None:
    print(f"Price elasticity data loaded: {df_elasticity.count()} products")
else:
    print("WARNING: No price elasticity data found - Phase 2 optimization will be limited")
    print(f"Run 10-ml-promotion-effectiveness.ipynb to generate {PRICE_ELASTICITY_TABLE_NAME}")

print()

In [ ]:
# =============================================================================
# CALCULATE PRODUCT AGE AND SALES VELOCITY
# =============================================================================

print("="*60)
print("CALCULATING PRODUCT AGE AND SALES VELOCITY")
print("="*60)

df_product_age = None
df_sales_velocity = None

if df_receipt_lines_source is not None:
    # Calculate days since first/last receipt and recent sales velocity
    df_receipt_lines = df_receipt_lines_source
    
    # Product age
    df_product_age = (
        df_receipt_lines
        .groupBy("product_id")
        .agg(
            F.min("event_date").alias("first_receipt_date"),
            F.max("event_date").alias("last_receipt_date")
        )
        .withColumn(
            "days_since_first_receipt",
            F.datediff(pricing_reference_date_lit, F.col("first_receipt_date"))
        )
        .withColumn(
            "days_since_last_receipt",
            F.datediff(pricing_reference_date_lit, F.col("last_receipt_date"))
        )
        .select("product_id", "days_since_first_receipt", "days_since_last_receipt")
    )
    print(f"Product age calculated for {df_product_age.count()} products")
    
    # Sales velocity over the configured sales window
    sales_window_start = F.date_sub(pricing_reference_date_lit, SALES_WINDOW_DAYS)
    df_sales_velocity = (
        df_receipt_lines
        .filter(F.col("event_date") >= sales_window_start)
        .groupBy("product_id")
        .agg(
            F.sum("quantity").alias("units_sold_in_window"),
            F.countDistinct("event_date").alias("days_with_sales")
        )
        .withColumn(
            "avg_daily_sales",
            F.col("units_sold_in_window") / F.lit(float(SALES_WINDOW_DAYS))
        )
        .select("product_id", "units_sold_in_window", "avg_daily_sales")
    )
    print(f"Sales velocity calculated for {df_sales_velocity.count()} products")
else:
    print("No receipt data - product age and sales velocity will be 0")

print()

In [ ]:
# =============================================================================
# PHASE 1: RULE-BASED MARKDOWN LOGIC
# =============================================================================

print("="*60)
print("PHASE 1: APPLYING RULE-BASED MARKDOWN LOGIC")
print("="*60)

# Start with product master
df_pricing = df_products

# Join with inventory if available
if df_inventory is not None:
    df_pricing = df_pricing.join(df_inventory, "product_id", "left")
else:
    df_pricing = df_pricing.withColumn("total_inventory", F.lit(None).cast("long"))
    df_pricing = df_pricing.withColumn("inventory_as_of", F.lit(None).cast("timestamp"))

# Join with product age if available
if df_product_age is not None:
    df_pricing = df_pricing.join(df_product_age, "product_id", "left")
else:
    df_pricing = df_pricing.withColumn("days_since_first_receipt", F.lit(0))
    df_pricing = df_pricing.withColumn("days_since_last_receipt", F.lit(0))

# Join with sales velocity if available
if df_sales_velocity is not None:
    df_pricing = df_pricing.join(df_sales_velocity, "product_id", "left")
else:
    df_pricing = df_pricing.withColumn("units_sold_in_window", F.lit(0))
    df_pricing = df_pricing.withColumn("avg_daily_sales", F.lit(0.0))

df_pricing = df_pricing.fillna({
    "total_inventory": 0,
    "days_since_first_receipt": 0,
    "days_since_last_receipt": 0,
    "units_sold_in_window": 0,
    "avg_daily_sales": 0.0,
})

# Initialize markdown factors and reason codes
df_pricing = df_pricing.withColumn("markdown_factor", F.lit(0.0))
df_pricing = df_pricing.withColumn("reason_codes", F.array())

# Rule 1: Age-based markdowns
df_pricing = df_pricing.withColumn(
    "age_markdown",
    F.when(
        F.col("days_since_last_receipt") >= AGE_THRESHOLD_AGGRESSIVE,
        F.lit(AGE_MARKDOWN_AGGRESSIVE)
    ).when(
        F.col("days_since_last_receipt") >= AGE_THRESHOLD_MODERATE,
        F.lit(AGE_MARKDOWN_MODERATE)
    ).otherwise(F.lit(0.0))
)

df_pricing = df_pricing.withColumn(
    "markdown_factor",
    F.greatest(F.col("markdown_factor"), F.col("age_markdown"))
)

df_pricing = df_pricing.withColumn(
    "reason_codes",
    F.when(
        F.col("age_markdown") == AGE_MARKDOWN_AGGRESSIVE,
        F.array_union(F.col("reason_codes"), F.array(F.lit("AGE_AGGRESSIVE")))
    ).when(
        F.col("age_markdown") == AGE_MARKDOWN_MODERATE,
        F.array_union(F.col("reason_codes"), F.array(F.lit("AGE_MODERATE")))
    ).otherwise(F.col("reason_codes"))
)

# Rule 2: Inventory-based markdowns
df_pricing = df_pricing.withColumn(
    "inventory_markdown",
    F.when(
        F.col("total_inventory") >= INVENTORY_HIGH_THRESHOLD,
        F.lit(INVENTORY_MARKDOWN)
    ).otherwise(F.lit(0.0))
)

df_pricing = df_pricing.withColumn(
    "markdown_factor",
    F.greatest(F.col("markdown_factor"), F.col("inventory_markdown"))
)

df_pricing = df_pricing.withColumn(
    "reason_codes",
    F.when(
        F.col("inventory_markdown") > 0,
        F.array_union(F.col("reason_codes"), F.array(F.lit("INVENTORY_HIGH")))
    ).otherwise(F.col("reason_codes"))
)

# Rule 3: Seasonal clearance using configured months and clearance tags
current_month = pricing_reference_date.month
is_clearance_period = current_month in CLEARANCE_MONTHS
clearance_tag_match = F.lit(False)
for clearance_tag in CLEARANCE_TAGS:
    clearance_tag_match = clearance_tag_match | F.lower(F.col("Tags")).contains(clearance_tag)

if is_clearance_period and CLEARANCE_TAGS:
    df_pricing = df_pricing.withColumn(
        "seasonal_markdown",
        F.when(
            (F.col("Tags").isNotNull()) & 
            clearance_tag_match,
            F.lit(CLEARANCE_MARKDOWN)
        ).otherwise(F.lit(0.0))
    )
    
    df_pricing = df_pricing.withColumn(
        "markdown_factor",
        F.greatest(F.col("markdown_factor"), F.col("seasonal_markdown"))
    )
    
    df_pricing = df_pricing.withColumn(
        "reason_codes",
        F.when(
            F.col("seasonal_markdown") > 0,
            F.array_union(F.col("reason_codes"), F.array(F.lit("SEASONAL_CLEARANCE")))
        ).otherwise(F.col("reason_codes"))
    )

# Calculate rule-based recommended price
df_pricing = df_pricing.withColumn(
    "rule_based_price",
    F.col("SalePrice") * (1 - F.col("markdown_factor"))
)

print(f"Rule-based markdown logic applied to {df_pricing.count()} products")
print()

In [ ]:
# =============================================================================
# PHASE 2: ELASTICITY-BASED OPTIMIZATION
# =============================================================================

print("="*60)
print("PHASE 2: APPLYING ELASTICITY-BASED OPTIMIZATION")
print("="*60)

if df_elasticity is not None:
    # Join elasticity data
    df_pricing = df_pricing.join(
        df_elasticity.select(
            "product_id",
            "elasticity_coefficient",
            "elasticity_category",
            "n_observations",
            F.col("confidence_interval_lower").alias("elasticity_ci_lower"),
            F.col("confidence_interval_upper").alias("elasticity_ci_upper")
        ),
        "product_id",
        "left"
    )
    
    # Calculate a Lerner-style candidate price from elasticity when demand is elastic enough.
    # For constant elasticity demand, the unconstrained revenue-maximizing markup exists only when |elasticity| > 1.
    df_pricing = df_pricing.withColumn(
        "elasticity_optimal_price",
        F.when(
            (F.col("elasticity_coefficient").isNotNull()) & 
            (F.col("elasticity_coefficient") < -1.0) &
            (F.col("n_observations") >= MIN_OBSERVATIONS_FOR_ELASTICITY),
            F.col("Cost") * (
                F.abs(F.col("elasticity_coefficient")) /
                (F.abs(F.col("elasticity_coefficient")) - F.lit(1.0))
            )
        ).otherwise(F.lit(None))
    )
    
    # Blend rule-based and elasticity-based prices
    df_pricing = df_pricing.withColumn(
        "blended_price",
        F.when(
            F.col("elasticity_optimal_price").isNotNull(),
            (F.col("elasticity_optimal_price") * ELASTICITY_WEIGHT) + 
            (F.col("rule_based_price") * (1 - ELASTICITY_WEIGHT))
        ).otherwise(F.col("rule_based_price"))
    )
    
    # Add elasticity reason code
    df_pricing = df_pricing.withColumn(
        "reason_codes",
        F.when(
            F.col("elasticity_optimal_price").isNotNull(),
            F.array_union(F.col("reason_codes"), F.array(F.lit("ELASTICITY_OPTIMIZED")))
        ).otherwise(F.col("reason_codes"))
    )
    
    print(f"Elasticity optimization applied to products with sufficient data")
else:
    # No elasticity data - use rule-based price only
    df_pricing = df_pricing.withColumn("elasticity_coefficient", F.lit(None).cast("decimal(10,4)"))
    df_pricing = df_pricing.withColumn("elasticity_category", F.lit(None).cast("string"))
    df_pricing = df_pricing.withColumn("n_observations", F.lit(None).cast("int"))
    df_pricing = df_pricing.withColumn("elasticity_ci_lower", F.lit(None).cast("decimal(10,4)"))
    df_pricing = df_pricing.withColumn("elasticity_ci_upper", F.lit(None).cast("decimal(10,4)"))
    df_pricing = df_pricing.withColumn("elasticity_optimal_price", F.lit(None).cast("decimal(10,2)"))
    df_pricing = df_pricing.withColumn("blended_price", F.col("rule_based_price"))
    print("No elasticity data available - using rule-based prices only")

print()

In [ ]:
# =============================================================================
# PHASE 2: INVENTORY URGENCY ADJUSTMENT
# =============================================================================

print("="*60)
print("PHASE 2: APPLYING INVENTORY URGENCY ADJUSTMENT")
print("="*60)

# Calculate days of inventory on hand
df_pricing = df_pricing.withColumn(
    "days_of_inventory",
    F.when(
        (F.col("avg_daily_sales") > 0.1),  # Avoid division by very small numbers
        F.col("total_inventory") / F.col("avg_daily_sales")
    ).otherwise(F.lit(999))  # High value = no urgency
)

# Apply additional markdown pressure for high inventory / overstock conditions
df_pricing = df_pricing.withColumn(
    "inventory_pressure_ratio",
    F.greatest(
        F.when(
            F.col("total_inventory") >= INVENTORY_HIGH_THRESHOLD,
            (F.col("total_inventory") / F.lit(float(INVENTORY_HIGH_THRESHOLD))) - F.lit(1.0)
        ).otherwise(F.lit(0.0)),
        F.when(
            F.col("total_inventory").isNotNull() & (F.col("days_of_inventory") >= INVENTORY_URGENCY_DAYS),
            (F.col("days_of_inventory") / F.lit(float(INVENTORY_URGENCY_DAYS))) - F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )
)

df_pricing = df_pricing.withColumn(
    "urgency_markdown",
    F.when(
        F.col("inventory_pressure_ratio") > 0,
        F.least(
            F.lit(0.15),
            F.lit(0.05) + (F.lit(0.10) * F.least(F.lit(1.0), F.col("inventory_pressure_ratio")))
        )
    ).otherwise(F.lit(0.0))
)

# Apply urgency adjustment to blended price
df_pricing = df_pricing.withColumn(
    "recommended_price_raw",
    F.col("blended_price") * (1 - F.col("urgency_markdown"))
)

# Add urgency reason code
df_pricing = df_pricing.withColumn(
    "reason_codes",
    F.when(
        F.col("urgency_markdown") > 0,
        F.array_union(F.col("reason_codes"), F.array(F.lit("INVENTORY_URGENCY")))
    ).otherwise(F.col("reason_codes"))
)

print(f"Inventory urgency adjustment applied")
print()

In [ ]:
# =============================================================================
# APPLY BUSINESS CONSTRAINTS
# =============================================================================

print("="*60)
print("APPLYING BUSINESS CONSTRAINTS")
print("="*60)

df_pricing = df_pricing.withColumn("recommendation_ts", F.current_timestamp())
df_pricing = df_pricing.withColumn("recommendation_date", F.current_date())

# Constraint 1: Minimum margin based on MIN_MARGIN_PCT
df_pricing = df_pricing.withColumn(
    "min_price",
    F.col("Cost") * (1 + MIN_MARGIN_PCT)
)

# Constraint 2: Maximum price (cannot exceed MSRP)
df_pricing = df_pricing.withColumn(
    "max_price",
    F.col("MSRP") * MAX_PRICE_FACTOR
)

# Apply min/max constraints
df_pricing = df_pricing.withColumn(
    "recommended_price",
    F.greatest(
        F.col("min_price"),
        F.least(F.col("recommended_price_raw"), F.col("max_price"))
    )
)

# Track constraint violations
df_pricing = df_pricing.withColumn(
    "hit_min_margin",
    F.col("recommended_price_raw") < F.col("min_price")
)

df_pricing = df_pricing.withColumn(
    "hit_max_price",
    F.col("recommended_price_raw") > F.col("max_price")
)

# Add constraint violation to reason codes
df_pricing = df_pricing.withColumn(
    "reason_codes",
    F.when(
        F.col("hit_min_margin"),
        F.array_union(F.col("reason_codes"), F.array(F.lit("CONSTRAINT_MIN_MARGIN")))
    ).otherwise(F.col("reason_codes"))
)

df_pricing = df_pricing.withColumn(
    "reason_codes",
    F.when(
        F.col("hit_max_price"),
        F.array_union(F.col("reason_codes"), F.array(F.lit("CONSTRAINT_MAX_PRICE")))
    ).otherwise(F.col("reason_codes"))
)

# Constraint 3: Max change per week (parameterized)
if df_previous_prices is not None:
    previous_recommendation_ts_col = (
        F.col("recommendation_ts").cast("timestamp").alias("previous_ts")
        if "recommendation_ts" in df_previous_prices.columns
        else F.to_timestamp("recommendation_date").alias("previous_ts")
    )
    previous_recommendation_date_col = (
        F.col("recommendation_date").alias("previous_recommendation_date")
        if "recommendation_date" in df_previous_prices.columns
        else F.to_date("recommendation_ts").alias("previous_recommendation_date")
    )
    df_prev = df_previous_prices.select(
        "product_id",
        F.col("recommended_price").alias("previous_price"),
        previous_recommendation_ts_col,
        previous_recommendation_date_col
    )
    
    df_pricing = df_pricing.join(df_prev, "product_id", "left")
    
    df_pricing = df_pricing.withColumn(
        "days_since_last_change",
        F.datediff(F.col("recommendation_date"), F.col("previous_recommendation_date"))
    )
    
    # Check if change exceeds MAX_CHANGE_PCT_WEEKLY and duration < MIN_DURATION_DAYS
    df_pricing = df_pricing.withColumn(
        "change_pct",
        F.when(
            F.col("previous_price").isNotNull() & (F.col("previous_price") != 0),
            F.abs((F.col("recommended_price") - F.col("previous_price")) / F.col("previous_price"))
        ).otherwise(F.lit(None).cast("double"))
    )
    
    df_pricing = df_pricing.withColumn(
        "violates_max_change",
        (F.col("change_pct") > MAX_CHANGE_PCT_WEEKLY) & 
        (F.col("days_since_last_change") < MIN_DURATION_DAYS)
    )
    
    # If violates, keep previous price
    df_pricing = df_pricing.withColumn(
        "recommended_price",
        F.when(
            F.col("violates_max_change") & F.col("previous_price").isNotNull(),
            F.col("previous_price")
        ).otherwise(F.col("recommended_price"))
    )
    
    df_pricing = df_pricing.withColumn(
        "reason_codes",
        F.when(
            F.col("violates_max_change"),
            F.array_union(F.col("reason_codes"), F.array(F.lit("CONSTRAINT_MAX_CHANGE")))
        ).otherwise(F.col("reason_codes"))
    )
else:
    df_pricing = df_pricing.withColumn("previous_price", F.lit(None).cast("decimal(10,2)"))
    df_pricing = df_pricing.withColumn("previous_ts", F.lit(None).cast("timestamp"))
    df_pricing = df_pricing.withColumn("previous_recommendation_date", F.lit(None).cast("date"))
    df_pricing = df_pricing.withColumn("days_since_last_change", F.lit(None).cast("int"))
    df_pricing = df_pricing.withColumn("change_pct", F.lit(None).cast("decimal(5,2)"))
    df_pricing = df_pricing.withColumn("violates_max_change", F.lit(False))

print(f"Constraints applied to {df_pricing.count()} products")
print()

In [ ]:
# =============================================================================
# PHASE 2: CALCULATE REVENUE IMPACT AND CONFIDENCE
# =============================================================================

print("="*60)
print("PHASE 2: CALCULATING REVENUE IMPACT AND CONFIDENCE")
print("="*60)

# Calculate expected revenue impact
# Formula: revenue_impact = (new_price - old_price) * expected_quantity
# where expected_quantity considers elasticity if available

df_pricing = df_pricing.withColumn(
    "price_change_pct",
    F.round((F.col("recommended_price") - F.col("SalePrice")) / F.col("SalePrice") * 100, 2)
)

# Estimate quantity change using elasticity
df_pricing = df_pricing.withColumn(
    "expected_quantity_change_pct",
    F.when(
        F.col("elasticity_coefficient").isNotNull(),
        F.col("elasticity_coefficient") * F.col("price_change_pct")
    ).otherwise(F.lit(0.0))  # Assume no quantity change if no elasticity data
)

df_pricing = df_pricing.withColumn(
    "expected_daily_quantity",
    F.greatest(
        F.lit(0.0),
        F.col("avg_daily_sales") * (1 + F.col("expected_quantity_change_pct") / 100)
    )
)

# Calculate daily revenue impact
df_pricing = df_pricing.withColumn(
    "current_daily_revenue",
    F.col("SalePrice") * F.col("avg_daily_sales")
)

df_pricing = df_pricing.withColumn(
    "expected_daily_revenue",
    F.col("recommended_price") * F.col("expected_daily_quantity")
)

df_pricing = df_pricing.withColumn(
    "daily_revenue_impact",
    F.col("expected_daily_revenue") - F.col("current_daily_revenue")
)

# Calculate 30-day revenue impact projection
df_pricing = df_pricing.withColumn(
    "projected_revenue_impact_30d",
    F.col("daily_revenue_impact") * 30
)

# Determine confidence level
df_pricing = df_pricing.withColumn(
    "confidence_level",
    F.when(
        (F.col("n_observations") >= 50) & 
        (F.col("elasticity_coefficient").isNotNull()) &
        (F.col("avg_daily_sales") >= 1.0),
        "HIGH"
    ).when(
        (F.col("n_observations") >= MIN_OBSERVATIONS_FOR_ELASTICITY) & 
        (F.col("elasticity_coefficient").isNotNull()),
        "MEDIUM"
    ).otherwise("LOW")
)

# Calculate confidence score (0-1)
df_pricing = df_pricing.withColumn(
    "confidence_score",
    F.when(
        F.col("confidence_level") == "HIGH", F.lit(0.85)
    ).when(
        F.col("confidence_level") == "MEDIUM", F.lit(0.60)
    ).otherwise(F.lit(0.35))
)

# Adjust confidence based on elasticity CI width
df_pricing = df_pricing.withColumn(
    "elasticity_ci_width",
    F.when(
        (F.col("elasticity_ci_upper").isNotNull()) & (F.col("elasticity_ci_lower").isNotNull()),
        F.col("elasticity_ci_upper") - F.col("elasticity_ci_lower")
    ).otherwise(F.lit(None))
)

df_pricing = df_pricing.withColumn(
    "confidence_score",
    F.when(
        (F.col("elasticity_ci_width").isNotNull()) & (F.col("elasticity_ci_width") > 1.0),
        F.col("confidence_score") * 0.9  # Reduce confidence for wide CI
    ).otherwise(F.col("confidence_score"))
)

print(f"Revenue impact and confidence calculated")
print()

In [ ]:
# =============================================================================
# CREATE OUTPUT TABLE
# =============================================================================

print("="*60)
print("CREATING PRICING RECOMMENDATIONS OUTPUT")
print("="*60)

# Add no-change reason for products that don't need markdown
df_pricing = df_pricing.withColumn(
    "reason_codes",
    F.when(
        F.size(F.col("reason_codes")) == 0,
        F.array(F.lit("NO_CHANGE"))
    ).otherwise(F.col("reason_codes"))
)

# Create final output with Phase 2 enhancements
df_output = df_pricing.select(
    "product_id",
    "ProductName",
    "Department",
    "Category",
    "Subcategory",
    F.col("Cost").alias("cost"),
    F.col("MSRP").alias("msrp"),
    F.col("SalePrice").alias("current_price"),
    F.col("recommended_price"),
    F.col("price_change_pct").alias("change_pct"),
    F.col("reason_codes"),
    F.col("markdown_factor"),
    F.col("total_inventory"),
    F.col("days_since_last_receipt"),
    F.col("avg_daily_sales"),
    F.col("days_of_inventory"),
    F.col("hit_min_margin"),
    F.col("hit_max_price"),
    F.col("violates_max_change"),
    # Phase 2 fields
    F.col("elasticity_coefficient"),
    F.col("elasticity_category"),
    F.col("projected_revenue_impact_30d"),
    F.col("confidence_level"),
    F.col("confidence_score"),
    F.when(
        F.col("elasticity_optimal_price").isNotNull(),
        F.lit("ELASTICITY_OPTIMIZED")
    ).otherwise(F.lit("RULE_BASED")).alias("model_type"),
    F.col("confidence_score").alias("ml_confidence"),
    F.col("recommendation_date"),
    F.col("recommendation_ts"),
    F.lit("2.0").alias("schema_version")
)

save_gold(df_output, PRICING_RECOMMENDATIONS_TABLE)
print()

In [ ]:
# Log results to MLflow
with mlflow.start_run(
    run_name=f"pricing_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "rule_based_elasticity",
        "min_margin_pct": MIN_MARGIN_PCT,
        "max_price_factor": MAX_PRICE_FACTOR,
        "max_change_pct_weekly": MAX_CHANGE_PCT_WEEKLY,
        "elasticity_weight": ELASTICITY_WEIGHT,
    })

    recs_count = spark.table(PRICING_RECOMMENDATIONS_TABLE_NAME).count()
    mlflow.log_metrics({
        "pricing_recommendations": recs_count,
    })

    print(f"MLflow run: {run.info.run_id}")
    print(f"Recommendations: {recs_count}")


In [ ]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================

print("="*60)
print("PRICING RECOMMENDATIONS SUMMARY")
print("="*60)

total_products = df_output.count()
print(f"\nTotal products analyzed: {total_products}")

# Count by reason code
print("\nRecommendations by reason:")
df_reasons = df_output.select(F.explode("reason_codes").alias("reason"))
df_reasons.groupBy("reason").count().orderBy(F.desc("count")).show(truncate=False)

# Count constraint violations
print("\nConstraint violations:")
min_margin_hits = df_output.filter(F.col("hit_min_margin")).count()
max_price_hits = df_output.filter(F.col("hit_max_price")).count()
max_change_hits = df_output.filter(F.col("violates_max_change")).count()

print(f"  Min margin: {min_margin_hits} products")
print(f"  Max price: {max_price_hits} products")
print(f"  Max change/week: {max_change_hits} products")

# Price change distribution
print("\nPrice change distribution:")
df_output.select(
    F.avg("change_pct").alias("avg_change_pct"),
    F.min("change_pct").alias("min_change_pct"),
    F.max("change_pct").alias("max_change_pct")
).show()

# Products with significant markdowns
significant_markdowns = df_output.filter(F.col("change_pct") < -5).count()
print(f"\nProducts with >5% markdown: {significant_markdowns}")

# Phase 2: Elasticity-based recommendations
print("\n" + "="*60)
print("PHASE 2: ELASTICITY-BASED OPTIMIZATION SUMMARY")
print("="*60)

elasticity_used = df_output.filter(F.col("elasticity_coefficient").isNotNull()).count()
print(f"\nProducts with elasticity data: {elasticity_used} ({elasticity_used*100/total_products:.1f}%)")

print("\nConfidence level distribution:")
df_output.groupBy("confidence_level").count().orderBy(F.desc("count")).show(truncate=False)

print("\nElasticity category distribution:")
df_output.filter(F.col("elasticity_category").isNotNull()).groupBy("elasticity_category").count().orderBy(F.desc("count")).show(truncate=False)

print("\nRevenue impact summary (30-day projection):")
df_output.select(
    F.sum("projected_revenue_impact_30d").alias("total_revenue_impact"),
    F.avg("projected_revenue_impact_30d").alias("avg_revenue_impact_per_product"),
    F.max("projected_revenue_impact_30d").alias("max_revenue_impact")
).show()

print("\nTop 10 products by expected revenue impact:")
df_output.orderBy(F.desc("projected_revenue_impact_30d")).select(
    "product_id",
    "ProductName",
    "current_price",
    "recommended_price",
    "change_pct",
    "projected_revenue_impact_30d",
    "confidence_level"
).show(10, truncate=False)

print("\n" + "="*60)
print("DYNAMIC PRICING PHASE 2 COMPLETE")
print("="*60)
print("\nCompleted features:")
print("  ✓ Phase 1: Rule-based markdown engine")
print("  ✓ Phase 2: Price elasticity integration")
print("  ✓ Phase 2: Elasticity-based optimization")
print("  ✓ Phase 2: Inventory urgency adjustments")
print("  ✓ Phase 2: Revenue impact estimation")
print("  ✓ Phase 2: Confidence level scoring")
print("\nNext steps:")
print(f"  - Review recommendations in {PRICING_RECOMMENDATIONS_TABLE_NAME}")
print("  - Monitor revenue impact vs actuals")
print("  - Phase 3: Add competitor price intelligence")
print("  - Phase 3: Add multi-product bundle optimization")